### **A Workflow for RAG**

La figure ci-dessous illustre les étapes clés impliquées dans la conception d’une solution RAG à l’aide de LLM. Examinons chacune de ces étapes en détail.

## **Step 1: Creating a Vector Database (Indexing)**

Nous créons une **base de données vectorielle** en découpant le document d’entrée en chunks, puis en associant à chaque chunk un **vecteur** à l’aide d’un modèle d’embeddings. Le processus global est décrit dans la figure ci-dessous.

#### **Choosing an embedding model**

## **Problem Definition**

Obtenir des détails concernant le rapport financier de l'OCP pour l'année 2023

In [1]:
import json
import tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma

from dotenv.ipython import load_dotenv
import os

# **LLM**

In [2]:
load_dotenv(override=True)

True

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# **Implementing RAG**

## **1 - Loading the PDF, Chunking**

In [4]:
pdf_file = "./pdfs/Rapport Financier Annuel  OCP 2023.pdf"

In [5]:
pdf_loader = PyPDFLoader(pdf_file)

Ici, nous avons utilisé `PyPDFLoader` parce que nous travaillons avec un seul document. Supposons que nous devions traiter plusieurs documents répartis dans différents fichiers au sein d’un dossier. Dans ce cas, nous utiliserions `PyPDFDirectoryLoader` pour pointer vers ce dossier. Il chargerait alors chaque fichier, le découperait en chunks et stockerait ces chunks dans une liste. Ce processus consiste à parcourir chaque fichier du répertoire, à découper le fichier, puis à stocker les chunks.

In [6]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(path = "./pdfs")

Ici, nous avons utilisé `PyPDFDirectoryLoader` car nous travaillons avec plusieurs documents répartis dans différents fichiers au sein d’un dossier. Il charge alors chaque fichier, le découpe en segments (chunks) et stocke ces segments dans une liste. Ce processus consiste à parcourir chaque fichier du répertoire, à le fragmenter, puis à enregistrer les fragments.

S’il s’agissait d’un seul fichier, nous utiliserions alors `PyPDFLoader`.

In [7]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=300,
    chunk_overlap=20
)

In [8]:
chunks = loader.load_and_split(text_splitter)

In [10]:
len(chunks)

576

## **2 - Vector Store - ChromaDB, Embeddings**

In [11]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model='text-embedding-ada-002')

In [12]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="rapport_ocp_V2",
    persist_directory="./store"
)

In [13]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [14]:
retrieved_chunks=retriever.invoke("Quelles sont les performances financières de l'OCP en 2023?")

In [15]:
print(retrieved_chunks)

[Document(metadata={'source': 'pdfs\\Rapport Financier Annuel  OCP 2023.pdf', 'total_pages': 210, 'creator': 'Nitro Pro 13 (13.32.0.623)', 'moddate': '2024-04-30T08:36:20+01:00', 'producer': 'PyPDF', 'page_label': '136', 'page': 135, 'creationdate': '2024-04-29T21:13:52+01:00'}, page_content='RÉSULTATS DE L’EXERCICE 2023 \nD’OCP S.A'), Document(metadata={'moddate': '2024-04-30T08:36:20+01:00', 'total_pages': 210, 'page': 135, 'creationdate': '2024-04-29T21:13:52+01:00', 'producer': 'PyPDF', 'page_label': '136', 'creator': 'Nitro Pro 13 (13.32.0.623)', 'source': 'pdfs\\Rapport Financier Annuel  OCP 2023.pdf'}, page_content='RÉSULTATS DE L’EXERCICE 2023 \nD’OCP S.A'), Document(metadata={'creationdate': '2024-04-29T21:13:52+01:00', 'moddate': '2024-04-30T08:36:20+01:00', 'source': 'pdfs\\Rapport Financier Annuel  OCP 2023.pdf', 'producer': 'PyPDF', 'page': 135, 'creator': 'Nitro Pro 13 (13.32.0.623)', 'page_label': '136', 'total_pages': 210}, page_content='RÉSULTATS DE L’EXERCICE 2023 \nD

In [16]:
print(len(retrieved_chunks))

5


## **3 - RAG Q&A**

### **Prompt Design**

In [17]:
prompt_template = """
Answer the following question based only on provided context
The context is about OCP Annual Financial Report 2023
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS
<context>
 {context}
</context>
<question>
 {question}
</question>
"""

### **Retrieving the Relevant Documents**

In [18]:
user_input = "Quelles sont les performances financières de l'OCP en 2023?"

In [19]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [20]:
context_for_query

'RÉSULTATS DE L’EXERCICE 2023 \nD’OCP S.A. RÉSULTATS DE L’EXERCICE 2023 \nD’OCP S.A. RÉSULTATS DE L’EXERCICE 2023 \nD’OCP S.A. 1. Contexte de l’activité 03\n2. Activité durant l’année 2023 07\n3. Résultats de l’exercice 2023 d’OCP S.A 11. 1. Contexte de l’activité 03\n2. Activité durant l’année 2023 07\n3. Résultats de l’exercice 2023 d’OCP S.A 11'

In [21]:
# Here the length is 10 because, earlier we have declared k=10
len(relevant_document_chunks)

5

In [22]:
prompt = prompt_template.format(context=context_for_query, question=user_input)

In [23]:
print(prompt)


Answer the following question based only on provided context
The context is about OCP Annual Financial Report 2023
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS
<context>
 RÉSULTATS DE L’EXERCICE 2023 
D’OCP S.A. RÉSULTATS DE L’EXERCICE 2023 
D’OCP S.A. RÉSULTATS DE L’EXERCICE 2023 
D’OCP S.A. 1. Contexte de l’activité 03
2. Activité durant l’année 2023 07
3. Résultats de l’exercice 2023 d’OCP S.A 11. 1. Contexte de l’activité 03
2. Activité durant l’année 2023 07
3. Résultats de l’exercice 2023 d’OCP S.A 11
</context>
<question>
 Quelles sont les performances financières de l'OCP en 2023?
</question>



In [24]:
resp = llm.invoke(prompt)

In [25]:
from IPython.display import Markdown

In [26]:
print(display(Markdown(resp.content)))

JE NE SAIS PAS

None


### **Defining the RAG function for response**




In [27]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

In [28]:
response = RAG("État du résultat global consolidé")
print(display(Markdown(response)))

Le résultat global consolidé pour l'exercice 2023 est de 14 187 millions de dirhams, tandis que pour l'exercice 2022, il était de 27 629 millions de dirhams.

None


In [29]:
user_input = "j'ai fain et je veux manger quelque chose"
output = RAG(user_input)
print(output)

JE NE SAIS PAS


In [30]:
response = RAG("Chiffre d'affaire de l'OCP en 2023")
print(display(Markdown(response)))

Le chiffre d'affaires de l'OCP en 2023 est de 81 239 323 844 Dirhams.

None


In [31]:
response = RAG("Quelles sont les performances financières de l'OCP en 2023")
print(display(Markdown(response)))

JE NE SAIS PAS

None


## **4 - Evaluation**

Utilisons maintenant la méthode LLM-as-a-judge pour vérifier la qualité du système RAG selon deux paramètres : la récupération (retrieval) et la génération (generation). Nous illustrons cette évaluation à partir des réponses générées à la question de la section précédente.

Nous utilisons le même modèle pour l’évaluation ; en d’autres termes, ici le LLM s’auto-évalue en notant la qualité de sa propre performance dans la tâche.

In [32]:
user_input ="État du résultat global consolidé"

In [33]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [34]:
user_message_template = """
 ###Question
 {question}
 ###Context
 {context}
 ###Answer
 {answer}
"""

In [35]:
# Default answer for an RAG query
answer = RAG(user_input)
print(display(Markdown(answer)))

Le résultat global consolidé pour l'exercice 2023 est de 14 187 millions de dirhams, tandis que pour l'exercice 2022, il était de 27 629 millions de dirhams.

None


### 1. Groundness

In [36]:
groundedness_rater_system_message="""
Vous êtes chargé d’évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d’IA pour générer la réponse, ainsi qu’une réponse générée par l’IA à la question.

Dans l’entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par l’IA commencera par ###Answer.

Critères d’évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1 — La métrique n’est pas respectée du tout
2 — La métrique n’est respectée que dans une mesure limitée
3 — La métrique est respectée dans une bonne mesure
4 — La métrique est respectée en grande partie
5 — La métrique est entièrement respectée

Métrique :
La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions :

Écrivez d’abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d’évaluation et attribuer un score.
"""

In [37]:
groundness_checker = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [38]:
def evaluate(system_message,user_message_template, question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question)
    prompt = f"""
     {system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
    """
    juge_response= model.invoke(prompt)
    return juge_response.content


In [39]:
resp=evaluate(groundedness_rater_system_message, user_message_template, user_input)

In [40]:
print(display(Markdown(resp)))

Pour évaluer la réponse selon la métrique, voici les étapes nécessaires :

1. **Identifier les informations clés dans le contexte :**
   - Résultat global consolidé pour 2023 : 14 187 millions de dirhams.
   - Résultat global consolidé pour 2022 : 27 629 millions de dirhams.
   - Part du Groupe pour 2023 : 14 259 millions de dirhams.
   - Part du Groupe pour 2022 : 27 580 millions de dirhams.
   - Part des intérêts ne donnant pas le contrôle pour 2023 : (72) millions de dirhams.
   - Part des intérêts ne donnant pas le contrôle pour 2022 : 49 millions de dirhams.

2. **Comparer ces informations avec celles fournies dans la réponse :**
   - La réponse indique que le résultat global consolidé pour 2023 est de 14 187 millions de dirhams, ce qui correspond au contexte.
   - Pour 2022, la réponse mentionne 27 629 millions de dirhams, ce qui est également correct.
   - La part du Groupe pour 2023 est donnée comme 14 259 millions de dirhams, ce qui est exact.
   - Pour 2022, la part du Groupe est indiquée comme 27 580 millions de dirhams, ce qui est correct.
   - La part des intérêts ne donnant pas le contrôle pour 2023 est mentionnée comme (72) millions de dirhams, ce qui est correct.
   - Pour 2022, la part des intérêts ne donnant pas le contrôle est donnée comme 49 millions de dirhams, ce qui est exact.

3. **Vérifier si la réponse est dérivée uniquement des informations du contexte :**
   - La réponse reprend fidèlement les chiffres et les informations fournies dans le contexte sans ajouter d'informations externes ou incorrectes.

4. **Évaluer dans quelle mesure la métrique est respectée :**
   - La réponse est entièrement dérivée des informations présentées dans le contexte et ne contient aucune information incorrecte ou non pertinente.

Sur la base de cette analyse, la réponse respecte entièrement la métrique. Par conséquent, le score attribué est de 5.

None


# **Conclusion**
- Dans ce notebook, nous avons appris à créer une application basée sur la génération augmentée par récupération (Retrieval-Augmented Generation, RAG), capable d’effectuer des questions-réponses sur des articles de recherche afin de récupérer des informations plus rapidement, plus efficacement et avec davantage de précision.